# 08 — Threshold Selection

For each (split strategy, model, feature condition), this notebook selects the classification threshold that **maximizes F1 on the validation set**, using only artifacts already produced by earlier notebooks:

- No-`duration` ("realistic") models: reuses `results/depth/threshold_sensitivity.csv` from `07_depth_analysis.ipynb` directly.
- With-`duration` (leakage) models: the same threshold sweep (0.05–0.95, step 0.05) is computed fresh from `results/leakage_predictions.csv`, since Phase 7 only swept thresholds for the no-`duration` models.

The chosen thresholds and their validation metrics are written to `results/frozen_evaluation_config.json` — "frozen" meaning these are the exact thresholds to be used later if/when a test-set evaluation is finally run, decided now, on validation data only, so they cannot be tuned in response to test-set performance.

**No model is retrained, no test data is read, and no existing results file is modified** — only `results/depth/threshold_sensitivity.csv` and `results/leakage_predictions.csv` are read, and the single new JSON file is written.

In [ ]:
import os
import json

import numpy as np
import pandas as pd
from sklearn.metrics import (
    average_precision_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
)

RESULTS_DIR = "../results"
DEPTH_DIR = f"{RESULTS_DIR}/depth"

THRESHOLDS = np.round(np.arange(0.05, 1.00, 0.05), 2)
SPLITS = ["stratified", "chronological"]
REALISTIC_MODELS = ["logistic_regression", "random_forest"]

pd.set_option("display.max_columns", 100)

## Load existing artifacts (read-only)

In [ ]:
threshold_sensitivity_path = f"{DEPTH_DIR}/threshold_sensitivity.csv"
leakage_predictions_path = f"{RESULTS_DIR}/leakage_predictions.csv"

for path in [threshold_sensitivity_path, leakage_predictions_path]:
    if not os.path.exists(path):
        raise FileNotFoundError(f"{path} not found — run 06_leakage_experiment.ipynb and 07_depth_analysis.ipynb first.")

no_duration_threshold_df = pd.read_csv(threshold_sensitivity_path)
no_duration_threshold_df["feature_set"] = "no_duration"

with_duration_preds = pd.read_csv(leakage_predictions_path)

print("No-duration threshold table:", no_duration_threshold_df.shape)
print("With-duration predictions:", with_duration_preds.shape, with_duration_preds["model"].unique())

## Threshold sweep for the with-`duration` models

Same grid (0.05–0.95, step 0.05) and same methodology as Phase 7's no-`duration` sweep, applied to `results/leakage_predictions.csv`.

In [ ]:
with_duration_rows = []
for split_name in SPLITS:
    for model_name in REALISTIC_MODELS:
        subset = with_duration_preds[(with_duration_preds["split_strategy"] == split_name) &
                                      (with_duration_preds["model"] == model_name)]
        assert len(subset) > 0, f"No with-duration predictions for {split_name}/{model_name}"
        y_true_bin = (subset["y_true"].values == "yes").astype(int)
        proba = subset["y_proba_yes"].values

        for t in THRESHOLDS:
            y_pred_t = (proba >= t).astype(int)
            with_duration_rows.append({
                "split_strategy": split_name,
                "model": model_name,
                "threshold": t,
                "precision": precision_score(y_true_bin, y_pred_t, zero_division=0),
                "recall": recall_score(y_true_bin, y_pred_t, zero_division=0),
                "f1": f1_score(y_true_bin, y_pred_t, zero_division=0),
                "is_default_threshold": bool(np.isclose(t, 0.5)),
                "feature_set": "with_duration",
            })

with_duration_threshold_df = pd.DataFrame(with_duration_rows)
with_duration_threshold_df[with_duration_threshold_df["is_default_threshold"]]

## Select the F1-maximizing threshold per (split, model, feature condition)

For each of the 8 combinations (2 splits × 2 models × 2 feature conditions), we pick the threshold with the highest validation F1. If several thresholds tie for the maximum, we keep the **smallest** tied threshold — a lower threshold flags more cases as positive for the same F1, which is a more conservative (higher-recall-leaning) choice for a marketing-contact use case, and ties are broken deterministically rather than arbitrarily.

In [ ]:
all_threshold_df = pd.concat([
    no_duration_threshold_df[["split_strategy", "model", "feature_set", "threshold", "precision", "recall", "f1"]],
    with_duration_threshold_df[["split_strategy", "model", "feature_set", "threshold", "precision", "recall", "f1"]],
], ignore_index=True)

best_rows = []
for (split_name, model_name, feature_set), group in all_threshold_df.groupby(["split_strategy", "model", "feature_set"]):
    max_f1 = group["f1"].max()
    best_candidates = group[np.isclose(group["f1"], max_f1)].sort_values("threshold")
    best = best_candidates.iloc[0]
    best_rows.append({
        "split_strategy": split_name,
        "model": model_name,
        "feature_set": feature_set,
        "selected_threshold": float(best["threshold"]),
        "precision_at_selected_threshold": float(best["precision"]),
        "recall_at_selected_threshold": float(best["recall"]),
        "f1_at_selected_threshold": float(best["f1"]),
        "n_tied_thresholds": int(len(best_candidates)),
    })

best_threshold_df = pd.DataFrame(best_rows).sort_values(["split_strategy", "feature_set", "model"]).reset_index(drop=True)
best_threshold_df

## Add threshold-independent metrics and prevalence

Average Precision and ROC-AUC don't depend on a threshold, so we attach them (recomputed directly from the saved probabilities, for both feature conditions) alongside each selected-threshold row, together with validation prevalence and sample size — giving a complete picture of each configuration, not just its F1-optimal operating point.

In [ ]:
no_duration_preds_path = f"{RESULTS_DIR}/validation_predictions.csv"
if not os.path.exists(no_duration_preds_path):
    raise FileNotFoundError(f"{no_duration_preds_path} not found — run 05_model_development.ipynb first.")
no_duration_preds = pd.read_csv(no_duration_preds_path)

preds_by_feature_set = {"no_duration": no_duration_preds, "with_duration": with_duration_preds}

extra_rows = []
for _, row in best_threshold_df.iterrows():
    preds = preds_by_feature_set[row["feature_set"]]
    subset = preds[(preds["split_strategy"] == row["split_strategy"]) & (preds["model"] == row["model"])]
    y_true_bin = (subset["y_true"].values == "yes").astype(int)
    proba = subset["y_proba_yes"].values
    extra_rows.append({
        "n_validation": int(len(subset)),
        "val_prevalence": float(y_true_bin.mean()),
        "average_precision": float(average_precision_score(y_true_bin, proba)),
        "roc_auc": float(roc_auc_score(y_true_bin, proba)),
    })

extra_df = pd.DataFrame(extra_rows)
final_config_df = pd.concat([best_threshold_df.reset_index(drop=True), extra_df], axis=1)
final_config_df

## Save `results/frozen_evaluation_config.json`

A new file only — no existing result is modified.

In [ ]:
configurations = []
for _, row in final_config_df.iterrows():
    configurations.append({
        "split_strategy": row["split_strategy"],
        "model": row["model"],
        "feature_set": row["feature_set"],
        "n_validation": int(row["n_validation"]),
        "val_prevalence": float(row["val_prevalence"]),
        "selected_threshold": float(row["selected_threshold"]),
        "n_tied_thresholds_at_max_f1": int(row["n_tied_thresholds"]),
        "metrics_at_selected_threshold": {
            "precision": float(row["precision_at_selected_threshold"]),
            "recall": float(row["recall_at_selected_threshold"]),
            "f1": float(row["f1_at_selected_threshold"]),
        },
        "threshold_independent_metrics": {
            "average_precision": float(row["average_precision"]),
            "roc_auc": float(row["roc_auc"]),
        },
    })

frozen_config = {
    "description": "Validation-selected classification thresholds, frozen for future (not-yet-run) test-set evaluation.",
    "selection_criterion": "Threshold in [0.05, 0.95] (step 0.05) that maximizes F1 on validation data; ties broken by smallest threshold.",
    "threshold_grid": [float(t) for t in THRESHOLDS],
    "source_notebooks": [
        "05_model_development.ipynb (no_duration predictions)",
        "06_leakage_experiment.ipynb (with_duration predictions)",
        "07_depth_analysis.ipynb (no_duration threshold sweep)",
        "08_threshold_selection.ipynb (with_duration threshold sweep + selection)",
    ],
    "test_set_accessed": False,
    "models_retrained": False,
    "configurations": configurations,
}

config_path = f"{RESULTS_DIR}/frozen_evaluation_config.json"
with open(config_path, "w") as f:
    json.dump(frozen_config, f, indent=2)

print("Saved:", config_path)
frozen_config

## Summary

- Selected the F1-maximizing threshold (grid 0.05–0.95) for all 8 combinations: 2 splits × 2 models × {no_duration, with_duration}.
- With-`duration` thresholds were computed fresh from `results/leakage_predictions.csv` using the same sweep methodology Phase 7 used for the no-`duration` models.
- Each configuration's precision/recall/F1 at its selected threshold, plus threshold-independent Average Precision and ROC-AUC, validation prevalence, and sample size, were saved to `results/frozen_evaluation_config.json`.
- These thresholds are **frozen**: chosen entirely from validation data, with the explicit intent that they not be re-tuned once/if a test-set evaluation is eventually run.

No model was retrained, no test data was read, and no existing results file (`threshold_sensitivity.csv`, `leakage_predictions.csv`, `validation_predictions.csv`, or anything else) was modified — only read.